# Grid-Grouped Cross-Validation Pipeline

**Condition-Based Grouped Cross-Validation for Reviewer**

This notebook demonstrates the enhanced grouped CV methodology that isolates samples with similar experimental conditions.

## 1. Library Imports

In [1]:
import warnings
warnings.filterwarnings(action='ignore')

import pandas as pd
import numpy as np
import glob
import os
import joblib
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error

print('Libraries imported')

Libraries imported


## 2. Configuration

In [2]:
RANDOM_STATE = 52
N_SPLITS = 5
RANDOM_SEED = 42

# Dataset configuration (MODIFY THESE)
DATA_PATH = r'C:\Users\HWY\Desktop\Chen\CDs-BT上传.xlsx'
TARGET_COLUMN = 'QY'

print(f'Config: {N_SPLITS}-fold GroupKFold, seed={RANDOM_SEED}')

Config: 5-fold GroupKFold, seed=42


## 3. Data Loading

In [3]:
# Load data
try:
    df = pd.read_excel(DATA_PATH, header=1)
except:
    df = pd.read_excel(DATA_PATH, header=0)

import re
df.columns = [re.sub(r'\(.*?\)', '', str(c)).strip() for c in df.columns]

if TARGET_COLUMN not in df.columns:
    raise ValueError(f"Target column '{TARGET_COLUMN}' not found")

y = df[TARGET_COLUMN].copy()

exclude_cols = [TARGET_COLUMN, 'DOI', 'No', 'Raw material']
X = df.drop([c for c in exclude_cols if c in df.columns], axis=1).copy()

# Encode Type column
if 'Type' in X.columns:
    TYPE_MAP = {'straw': 1, 'wood': 2, 'manure': 3, 'food waste': 4,
                'sewage sludge': 5, 'algae': 6, 'by-product': 7, 'mixed': 8}
    X['Type'] = X['Type'].astype(str).str.strip().str.lower().map(TYPE_MAP)
    print('Type column encoded')

print(f'Data loaded: {len(df)} samples, {X.shape[1]} features')
print(f'Target: {TARGET_COLUMN} | Range: [{y.min():.2f}, {y.max():.2f}]')

Type column encoded
Data loaded: 172 samples, 16 features
Target: QY | Range: [0.18, 17.90]


## 4. Auto-Detect Reaction Features

In [4]:
# Auto-detect 3 or 4 reaction condition features
required_features = ['T', 'RT', 'SLR']

for feat in required_features:
    if feat not in df.columns:
        raise ValueError(f"Missing required feature: {feat}")

if 'Cycles' in df.columns and df['Cycles'].notna().sum() > 0:
    reaction_features = required_features + ['Cycles']
    print(f'Detected 4 reaction features: {reaction_features}')
    print('  (HC/AP dataset mode)')
else:
    reaction_features = required_features
    print(f'Detected 3 reaction features: {reaction_features}')
    print('  (CDs dataset mode)')

Detected 3 reaction features: ['T', 'RT', 'SLR']
  (CDs dataset mode)


## 5. Create Fixed-Interval Grid Configuration

In [5]:
# Auto-generate fixed interval bins
print('\nAuto-generated fixed interval configuration:')
print('-'*60)

bins_config = {}

for feat in reaction_features:
    feat_min = df[feat].min()
    feat_max = df[feat].max()
    
    if feat == 'T':
        start = int(feat_min // 20) * 20
        end = int(feat_max // 20 + 1) * 20 + 20
        bins_config['T'] = list(range(start, end, 20))
        print(f"  T: {len(bins_config['T'])-1} intervals, range [{start}, {end})")
    elif feat == 'RT':
        bins_config['RT'] = [0, 1, 2, 5]
        print(f"  RT: 3 intervals [0-1, 1-2, 2-5)")
    elif feat == 'SLR':
        bins_config['SLR'] = [0, 10, 30, 150]
        print(f"  SLR: 3 intervals [0-10, 10-30, 30-150)")
    elif feat == 'Cycles':
        bins_config['Cycles'] = [0, 1, 2, 5]
        print(f"  Cycles: 3 intervals [0-1, 1-2, 2-5)")

print('\nFixed intervals ensure no data leakage')


Auto-generated fixed interval configuration:
------------------------------------------------------------
  T: 8 intervals, range [120, 300)
  RT: 3 intervals [0-1, 1-2, 2-5)
  SLR: 3 intervals [0-10, 10-30, 30-150)

Fixed intervals ensure no data leakage


## 6. Create Grid Groups (No Data Leakage)

In [6]:
print('\n' + '='*60)
print('Fixed-Interval Grid Grouping')
print('='*60)

grid_features = list(bins_config.keys())
print(f'\nFeatures for grid: {grid_features}')

# Calculate theoretical max groups
total_bins = 1
for feat, edges in bins_config.items():
    total_bins *= (len(edges) - 1)
print(f'Theoretical max groups: {total_bins}')

# Extract grid features
X_grid = X[grid_features].copy()

# Handle missing values (constant fill, no statistical learning)
missing_count = X_grid.isna().sum().sum()
if missing_count > 0:
    print(f'\nDetected {missing_count} missing values')
    print('Filling with 0 (constant, no statistics learned)')
    X_grid_imputed = X_grid.fillna(0)
else:
    print('\nNo missing values, using raw data')
    X_grid_imputed = X_grid

print('\nFixed interval division for each feature:')
print('-'*60)

for feat, edges in bins_config.items():
    X_grid_imputed[f'{feat}_bin'] = pd.cut(
        X_grid_imputed[feat],
        bins=edges,
        labels=False,
        include_lowest=True
    )
    print(f'{feat}: {len(edges)-1} intervals')
    for i in range(min(len(edges)-1, 3)):
        print(f"  Interval {i}: [{edges[i]}, {edges[i+1]})")

# Combine bins into grid groups
grid_ids = X_grid_imputed[[f'{feat}_bin' for feat in grid_features]].astype(str)
grid_groups = grid_ids.apply(lambda row: '_'.join(row), axis=1)

# Convert to numeric IDs
unique_grids = grid_groups.unique()
grid_mapping = {grid: idx for idx, grid in enumerate(unique_grids)}
grid_groups_numeric = grid_groups.map(grid_mapping).values

print('\n' + '-'*60)
print(f'Actual grid groups created: {len(unique_grids)}')

unique, counts = np.unique(grid_groups_numeric, return_counts=True)
print(f'\nSamples per grid group:')
print(f"  Min: {counts.min()}")
print(f"  Max: {counts.max()}")
print(f"  Median: {np.median(counts):.0f}")
print(f"  Mean: {counts.mean():.1f}")

print('\nGrid grouping complete (no data leakage)')


Fixed-Interval Grid Grouping

Features for grid: ['T', 'RT', 'SLR']
Theoretical max groups: 72

Detected 3 missing values
Filling with 0 (constant, no statistics learned)

Fixed interval division for each feature:
------------------------------------------------------------
T: 8 intervals
  Interval 0: [120, 140)
  Interval 1: [140, 160)
  Interval 2: [160, 180)
RT: 3 intervals
  Interval 0: [0, 1)
  Interval 1: [1, 2)
  Interval 2: [2, 5)
SLR: 3 intervals
  Interval 0: [0, 10)
  Interval 1: [10, 30)
  Interval 2: [30, 150)

------------------------------------------------------------
Actual grid groups created: 35

Samples per grid group:
  Min: 1
  Max: 27
  Median: 2
  Mean: 4.9

Grid grouping complete (no data leakage)


## 7. Load Best Model Parameters

In [7]:
# Auto-find best model
basename = os.path.basename(DATA_PATH).upper()

if 'HC' in basename:
    dataset_prefix = 'HC_Yield'
    model_dir = r'C:\Users\HWY\Desktop\Chen\水热炭化-HC\HC-模型输出'
elif 'AP' in basename:
    dataset_prefix = 'AP_TN'
    model_dir = r'C:\Users\HWY\Desktop\Chen\水热炭化-AP\AP-模型输出'
elif 'CDS' in basename or 'CD' in basename:
    dataset_prefix = 'CDs_QY'
    model_dir = r'C:\Users\HWY\Desktop\Chen\水热炭化-CDs\CDs-模型输出'
else:
    raise ValueError(f"Cannot identify dataset type from: {DATA_PATH}")

pkl_pattern = os.path.join(model_dir, f'{dataset_prefix}_*_best_model.pkl')
pkl_files = glob.glob(pkl_pattern)

if len(pkl_files) == 0:
    raise FileNotFoundError(f"No model file found matching '{dataset_prefix}_*_best_model.pkl'")

model_path = max(pkl_files, key=os.path.getmtime)
best_name = os.path.basename(model_path).replace(f'{dataset_prefix}_', '').replace('_best_model.pkl', '')

print(f'Auto-identified best model: {best_name}')
print(f'Model path: {model_path}')

# Load model and extract parameters
try:
    final_pipeline = joblib.load(model_path)
    gbr = final_pipeline.named_steps['reg']
    params = {
        'n_estimators': gbr.n_estimators,
        'learning_rate': gbr.learning_rate,
        'max_depth': gbr.max_depth,
        'min_samples_split': gbr.min_samples_split,
        'min_samples_leaf': gbr.min_samples_leaf,
        'subsample': gbr.subsample,
        'max_features': gbr.max_features,
        'random_state': gbr.random_state
    }
    print('\nExtracted parameters:')
    for k, v in params.items():
        print(f"  {k}: {v}")
except Exception as e:
    print(f'Warning: Could not load model: {str(e)[:50]}...')
    print('Using default parameters')
    params = {
        'n_estimators': 610, 'learning_rate': 0.048, 'max_depth': 10,
        'min_samples_split': 2, 'min_samples_leaf': 2,
        'subsample': 0.75, 'max_features': 0.30, 'random_state': 52
    }

Auto-identified best model: GBDT
Model path: C:\Users\HWY\Desktop\Chen\水热炭化-CDs\CDs-模型输出\CDs_QY_GBDT_best_model.pkl

Extracted parameters:
  n_estimators: 641
  learning_rate: 0.02008815155266471
  max_depth: 5
  min_samples_split: 12
  min_samples_leaf: 1
  subsample: 0.9087096825400937
  max_features: 0.632781916203825
  random_state: 52


## 8. Grid-Grouped Cross-Validation

In [8]:
print('\n' + '='*60)
print('Grid-Grouped Cross-Validation')
print('='*60)

unique_groups = np.unique(grid_groups_numeric)
n_groups = len(unique_groups)
print(f'\nTotal grid groups: {n_groups}')

if n_groups < N_SPLITS:
    print(f'WARNING: Groups ({n_groups}) < folds ({N_SPLITS}), adjusting to {n_groups} folds')
    N_SPLITS = n_groups

# Fix group order with random seed
np.random.seed(RANDOM_SEED)
shuffled_groups = np.random.permutation(unique_groups)
group_mapping = {old_id: new_id for new_id, old_id in enumerate(shuffled_groups)}
grid_groups_fixed = np.array([group_mapping[g] for g in grid_groups_numeric])

print(f'Using random seed {RANDOM_SEED} for reproducibility')

gkf = GroupKFold(n_splits=N_SPLITS)

fold_results = []
oof_predictions = pd.Series(index=y.index, dtype=float)

print(f'\nStarting {N_SPLITS}-fold CV...')
print('-'*60)

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=grid_groups_fixed), 1):
    X_train_fold = X.iloc[train_idx].copy()
    X_test_fold = X.iloc[test_idx].copy()
    y_train_fold = y.iloc[train_idx].copy()
    y_test_fold = y.iloc[test_idx].copy()
    
    test_groups = grid_groups_fixed[test_idx]
    n_test_groups = len(np.unique(test_groups))
    
    # Pipeline with imputer and scaler (fit only on training fold)
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('reg', GradientBoostingRegressor(**params))
    ])
    
    pipeline.fit(X_train_fold, y_train_fold)
    y_pred_fold = pipeline.predict(X_test_fold)
    oof_predictions.iloc[test_idx] = y_pred_fold
    
    r2_fold = r2_score(y_test_fold, y_pred_fold)
    rmse_fold = np.sqrt(mean_squared_error(y_test_fold, y_pred_fold))
    mae_fold = np.mean(np.abs(y_test_fold - y_pred_fold))
    
    fold_results.append({
        'Fold': fold,
        'n_grids_test': n_test_groups,
        'n_test': len(test_idx),
        'n_train': len(train_idx),
        'R2': r2_fold,
        'RMSE': rmse_fold,
        'MAE': mae_fold
    })
    
    print(f"Fold {fold}: {n_test_groups} grids | n_test={len(test_idx):3d} | R2={r2_fold:.4f} | RMSE={rmse_fold:.4f}")

print('-'*60)
pooled_r2 = r2_score(y, oof_predictions)
pooled_rmse = np.sqrt(mean_squared_error(y, oof_predictions))
pooled_mae = np.mean(np.abs(y - oof_predictions))

print('\n' + '='*60)
print('Out-of-Fold Summary')
print('='*60)
print(f"  Pooled R2:   {pooled_r2:.4f}")
print(f"  Pooled RMSE: {pooled_rmse:.4f}")
print(f"  Pooled MAE:  {pooled_mae:.4f}")
print('='*60)


Grid-Grouped Cross-Validation

Total grid groups: 35
Using random seed 42 for reproducibility

Starting 5-fold CV...
------------------------------------------------------------
Fold 1: 6 grids | n_test= 35 | R2=0.6257 | RMSE=1.3166
Fold 2: 7 grids | n_test= 35 | R2=-0.1081 | RMSE=4.5020
Fold 3: 6 grids | n_test= 34 | R2=0.7407 | RMSE=0.9712
Fold 4: 7 grids | n_test= 34 | R2=0.3856 | RMSE=0.9365
Fold 5: 9 grids | n_test= 34 | R2=0.7361 | RMSE=2.0151
------------------------------------------------------------

Out-of-Fold Summary
  Pooled R2:   0.5185
  Pooled RMSE: 2.3748
  Pooled MAE:  1.2390


## 9. Results Summary

In [9]:
results_df = pd.DataFrame(fold_results)

print('\nDetailed fold results:')
print(results_df.to_string(index=False))

print(f'\nBest model: {best_name}')
print(f'Final R2: {pooled_r2:.4f}')


Detailed fold results:
 Fold  n_grids_test  n_test  n_train        R2     RMSE      MAE
    1             6      35      137  0.625739 1.316563 0.867360
    2             7      35      137 -0.108075 4.501990 2.791378
    3             6      34      138  0.740731 0.971195 0.703552
    4             7      34      138  0.385621 0.936550 0.684318
    5             9      34      138  0.736095 2.015056 1.113714

Best model: GBDT
Final R2: 0.5185


## 10. Save Results

In [10]:
# Save results
base_dir = os.path.dirname(os.path.dirname(model_path))
OUTPUT_DIR = os.path.join(base_dir, f'{best_name}_grid_cv_results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

results_path = os.path.join(OUTPUT_DIR, 'fold_results.csv')
results_df.to_csv(results_path, index=False, encoding='utf-8-sig')
print(f'Fold results saved: {results_path}')

prediction_df = df.copy()
prediction_df['Measured'] = y
prediction_df['Predicted'] = oof_predictions
prediction_df['Residual'] = y - oof_predictions
prediction_df['Grid_Group'] = grid_groups_numeric
pred_path = os.path.join(OUTPUT_DIR, 'oof_predictions.csv')
prediction_df.to_csv(pred_path, index=False, encoding='utf-8-sig')
print(f'Predictions saved: {pred_path}')

print(f'\nAll results saved to: {OUTPUT_DIR}')

Fold results saved: C:\Users\HWY\Desktop\Chen\水热炭化-CDs\GBDT_grid_cv_results\fold_results.csv
Predictions saved: C:\Users\HWY\Desktop\Chen\水热炭化-CDs\GBDT_grid_cv_results\oof_predictions.csv

All results saved to: C:\Users\HWY\Desktop\Chen\水热炭化-CDs\GBDT_grid_cv_results
